# How much of the spread of kappa is accounted for?

**Status: post hoc synthesis, no new runs.** The ratio kappa/kappa_unif has median 0.81 and a 5-95% range 0.44-1.33 over stationary blob rules. kappa_unif is rule-independent (sd ln 0.004), so its spread is the spread of ln kappa. This notebook budgets that spread into start-to-start noise, variance predicted by the first eight rule letters, and the rest. It does not address why the *location* is about 0.8 x kappa_unif.

**Assumptions.** Predictors tied to kappa by the growth identity (c, ln A22, a, beta, fill) are excluded; ln A22 is the collider found in the c-law Stage 4. The prefix R^2 is against rule-level means and is scaled by the ICC to a share of single-run variance (valid if those means are noise-free).

In [1]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
d = json.loads((root / "experiments/archive/2026-10-kappa-spread-budget/results/spread_budget.json").read_text())
print("command:", d["provenance"]["command"])
for name, h in d["provenance"]["input_sha256"].items():
    print(h[:16], name)

command: uv run experiments/2026-10-20-kappa-spread-budget/scripts/spread_budget.py
ad7a062d72dd9390 experiments/2026-10-16-kappa-uniform-baseline/results/census_spread.txt
30b2c42476650971 experiments/2026-10-kappa-rule-space/results/w2_stationarity.txt
13f993ec0598cf76 experiments/2026-10-kappa-rule-space/results/x2_prefix_heldout.txt
c2dc0e8cf016a69a experiments/2026-10-kappa-rule-space/results/w3c_stats.txt


In [2]:
print(f"total sd ln kappa {d['sd_ln_kappa_census_stationary']:.3f} (n={d['n_census']}, variance {d['var_total']:.4f})")
print(f"start noise share {d['noise_share']:.3f} (ICC {d['icc_pool_stationary']:.3f}); noise sd {d['noise_sd']:.3f}, rule-mean sd {d['rule_mean_sd']:.3f}")
print()
print(f"{'prefix-8 R^2 source':26s} {'R^2':>6s} {'prefix':>7s} {'rule, unexplained':>18s} {'noise':>6s}")
for label, r in d["budget"].items():
    print(f"{label:26s} {r['r2_prefix8']:6.3f} {r['explained_of_total']:7.2f} {r['rule_unexplained_of_total']:18.2f} {r['noise_of_total']:6.2f}")

# agreement with README / results text
assert abs(d["sd_ln_kappa_census_stationary"] - 0.349) < 1e-3
assert abs(d["noise_share"] - 0.149) < 1e-3
b = d["budget"]
for r in b.values():
    assert abs(r["explained_of_total"] + r["rule_unexplained_of_total"] + r["noise_of_total"] - 1) < 1e-9
assert 0.46 < min(r["explained_of_total"] for r in b.values()) and max(r["explained_of_total"] for r in b.values()) < 0.55

total sd ln kappa 0.349 (n=2185, variance 0.1218)
start noise share 0.149 (ICC 0.851); noise sd 0.126, rule-mean sd 0.302

prefix-8 R^2 source           R^2  prefix  rule, unexplained  noise
pool patched means          0.552    0.47               0.38   0.15
fresh rules, strict         0.633    0.54               0.31   0.15


## What this shows and does not show

- About half the variance of ln kappa is predictable from the first eight letters, about a third is rule-determined but not captured by the prefix, and about a sixth is start-to-start noise.
- The unexplained rule-determined part has sd about 0.2 in ln kappa; a mechanism for the spread would have to act there.
- **Not shown:** the location (median about 0.8 x kappa_unif, below every walker reference), or a mechanism for any part.

## Caveats
Three different cohorts (census single runs, pool-192 patched starts, fresh 13-14 letter rules); no uncertainty is propagated (shares are rough to about +-0.1); prefix families overlap between training and test, so this is transfer within families, not to unseen ones.

## Next
A registered held-out comparison of prefix, suffix and composition features on the fresh rules, to say what the unexplained third consists of.